# Model Evaluation
This notebook evaluates the performance of our fine-tuned open-source model, which resulted in an error of $64.58

**Source Colab:** [Link](https://colab.research.google.com/drive/12OJ9NsZy_bWsFUwOicT0d7zljcVhvEUb#scrollTo=uuTX-xonNeOK)

In [ ]:
# Install dependencies if needed
# !pip install -q --upgrade bitsandbytes trl

In [ ]:
import os
import sys
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel

# Add parent directory to path to find util.py if it's in week7/ directory
sys.path.append(os.path.abspath("../../"))
try:
    from util import evaluate
except ImportError:
    print("Could not import evaluate from util. Ensure util.py is available in the expected path.")

In [ ]:
# Constants
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "ukazee"
DATA_USER = "ukazee"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite"
RUN_NAME = "2026-03-04_21.06.36-lite"
REVISION = None
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - QLoRA
QUANT_4_BIT = True
capability = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = capability[0] >= 8

In [ ]:
# Log in to HuggingFace
# Replace with your token or set HF_TOKEN environment variable
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)
else:
    print("HF_TOKEN environment variable not set. Please log in manually using huggingface-cli login or set the variable.")

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

In [ ]:
print(f"Test set size: {len(test)}")
print(test[0])

**Now load the Tokenizer and Model**

In [ ]:
# pick the right quantization
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# Load the Tokenizer and the Model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# Inspect the fine-tuned model structure
print(fine_tuned_model)

In [ ]:
def model_predict(item):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(item["prompt"], return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
set_seed(42)
evaluate(model_predict, test)